In [21]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# ==========================================================
# PROJECT PATH
# ==========================================================

PROJECT_DIR = Path(r"C:\Users\devar\OneDrive\Documents\Devarsh69\Devarsh.py")

DATA_PATH = PROJECT_DIR / "Data" / "online_retail_feature_engineered.csv"

MODEL_DIR = PROJECT_DIR / "airflow" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "churn_xgboost.pkl"

print("="*60)
print("Loading Dataset")
print("="*60)

print(DATA_PATH)

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv(DATA_PATH)

print("\nDataset Shape :", df.shape)

print("\nColumns:")
print(df.columns.tolist())

# ==========================================================
# CREATE CHURN LABEL
# ==========================================================

df["Churn"] = (df["Recency"] > 90).astype(int)

# ==========================================================
# FEATURES
# ==========================================================

features = [
    "Recency",
    "Frequency",
    "Monetary"
]

X = df[features]

y = df["Churn"]

# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

# ==========================================================
# TRAIN MODEL
# ==========================================================

model = XGBClassifier(

    random_state=42,

    eval_metric="logloss"

)

model.fit(

    X_train,

    y_train

)

# ==========================================================
# PREDICTION
# ==========================================================

pred = model.predict(X_test)

prob = model.predict_proba(X_test)[:,1]

# ==========================================================
# METRICS
# ==========================================================

print("\nAccuracy :", round(accuracy_score(y_test,pred),4))

print("Precision :", round(precision_score(y_test,pred),4))

print("Recall :", round(recall_score(y_test,pred),4))

print("ROC AUC :", round(roc_auc_score(y_test,prob),4))

print("\nConfusion Matrix")

print(confusion_matrix(y_test,pred))

# ==========================================================
# SAVE MODEL
# ==========================================================

joblib.dump(

    model,

    MODEL_PATH

)

print("\nModel Saved Successfully")

print(MODEL_PATH)

Loading Dataset
C:\Users\devar\OneDrive\Documents\Devarsh69\Devarsh.py\Data\online_retail_feature_engineered.csv

Dataset Shape : (663373, 29)

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Year', 'Month', 'Day', 'Weekday', 'TotalSales', 'InvoiceMonth', 'InvoiceYear', 'InvoiceDay', 'RollingSales_7', 'RollingQty_7', 'PurchaseCount', 'AverageSpend', 'LifetimeSales', 'CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score']

Accuracy : 1.0
Precision : 1.0
Recall : 1.0
ROC AUC : 1.0

Confusion Matrix
[[102606      0]
 [     0  30069]]

Model Saved Successfully
C:\Users\devar\OneDrive\Documents\Devarsh69\Devarsh.py\airflow\models\churn_xgboost.pkl


In [22]:
from airflow import DAG

from airflow.operators.python import PythonOperator

from datetime import datetime

from pathlib import Path

import subprocess

import pandas as pd

# ==========================================================
# PROJECT PATH
# ==========================================================

PROJECT_DIR = Path(r"C:\Users\devar\OneDrive\Documents\Devarsh69\Devarsh.py")

DATA_DIR = PROJECT_DIR / "Data"

DRIFT_FILE = DATA_DIR / "drift_summary.csv"

SCRIPT_FILE = PROJECT_DIR / "airflow" / "scripts" / "retrain_pipeline.py"

# ==========================================================
# CHECK DRIFT
# ==========================================================

def check_drift():

    print("="*60)
    print("Checking Data Drift")
    print("="*60)

    drift = pd.read_csv(DRIFT_FILE)

    print(drift)

    if drift["DriftDetected"].sum() > 0:

        print("\nData Drift Detected")

    else:

        raise Exception("No Data Drift Found")

# ==========================================================
# RETRAIN MODEL
# ==========================================================

def retrain_model():

    print("="*60)
    print("Retraining Model")
    print("="*60)

    subprocess.run(

        ["python", str(SCRIPT_FILE)],

        check=True

    )

# ==========================================================
# DAG
# ==========================================================

default_args = {

    "owner":"Devarsh",

    "start_date":datetime(2026,1,1)

}

with DAG(

    dag_id="online_retail_retraining",

    default_args=default_args,

    schedule="@weekly",

    catchup=False,

    tags=["Retail","MLOps"]

) as dag:

    drift = PythonOperator(

        task_id="check_drift",

        python_callable=check_drift

    )

    retrain = PythonOperator(

        task_id="retrain_model",

        python_callable=retrain_model

    )

    drift >> retrain

c:\Users\devar\AppData\Local\Programs\Python\Python312\Lib\site-packages\airflow\__init__.py:47: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(


2026-06-30T05:22:01.070076Z [warning  ] The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`. [py.warnings] category=DeprecatedImportWarning filename=C:\Users\devar\AppData\Local\Temp\ipykernel_3904\1136695811.py lineno=3
